# 00 · The question and the assets

**Goal:** distinguish a potentially useful teaching mechanism from
ordinary gains caused by additional training.

**Question:** can a short training response tell us which synthetic lesson
will improve an unfamiliar pose estimator on real video?

The source sequence is **00 → 01 → 02 → 03 → 07**. Run 02 once per configured
source student. After the source decision, prepare independent GAVD references
in 04, deploy without reading their labels in 05, and exchange selected lessons
in 08. Explicitly evaluate in 06. These notebooks contain no precomputed
research results or substitute models.

[Proposal](../../notes/research-agenda/proposals/synthetic-training-selection.md)
· [Notebook guide](README.md)
· [HAIC setup and launch commands](../../slurm/synthetic-training/README.md)

In [ ]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

root_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ.setdefault("GAVD6_ROOT", str(PROJECT_ROOT))
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, SVG, display
from gavd6_sjepa.research_directions.synthetic_training.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training import workflow

cfg = RunConfig.from_env()
RUN_ROOT = cfg.root
get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

def show_result(result):
    # Display the tables and artifact paths returned by a workflow stage.
    if isinstance(result, pd.DataFrame):
        display(result)
    elif isinstance(result, dict):
        for name, value in result.items():
            display(Markdown(f"### {name.replace('_', ' ')}"))
            if isinstance(value, pd.DataFrame):
                display(value)
            elif isinstance(value, Path) and value.suffix == ".svg" and value.is_file():
                display(SVG(filename=str(value)))
            else:
                print(json.dumps(value, indent=2, default=str) if isinstance(value, (list, dict)) else value)
    else:
        print(result)

print(f"Run: {RUN_ROOT}")
print(f"Context representation: {cfg.context_kind}; device: {cfg.device}")

## 1. Understand the experiment in one example

A pose estimator predicts twelve visible body landmarks. A **lesson**
is a small labeled set of rendered AMASS images, such as oblique views
or partly occluded people. A **probe** is a common short training update.
A **selector** uses the estimator's response to choose its next lesson.

We need four findings in sequence:

1. Synthetic lessons can improve real accuracy beyond equal-budget replay.
2. Different students or settings benefit from different lessons.
3. Target-video prediction changes select better lessons than current
   weaknesses and source learning progress alone.
4. A selector learned on source trials transfers to a held architecture.

Improved confidence or lower training loss does not establish real
accuracy. A useful nearest-neighbor selector is enough to test the
mechanism; a more complicated teacher is not itself the contribution.

In [ ]:
display(pd.DataFrame([
    ("train", "Fit source outcome predictors", "AMASS source references"),
    ("validation", "Choose selector settings and comparators", "Separate source students and references"),
    ("held", "Test transfer to an unseen architecture", "Never enters source fitting or model selection"),
], columns=["Student role", "Purpose", "Reference boundary"]))
display(pd.DataFrame(cfg.students))

## 2. Inspect actual asset availability

This reads the repository's AMASS and GAVD manifests and checks local
paths. It does not infer availability from a manifest count or claim
that a checkpoint has successfully loaded. The following stage needs
full-body AMASS files, licensed body models, compatible texture/UV
assets, photographic backgrounds, and labeled COCO replay data.

Model configuration files and their corresponding released weights
must be present. Use the HAIC guide's isolated MMPose environment when
the main project's PyTorch version is incompatible with MMCV.

In [ ]:
started = perf_counter()
inventory = workflow.inventory(cfg)
show_result(inventory)
print(f"Inventory took {perf_counter() - started:.1f} seconds.")

## 3. Keep the interpretation narrow

GAVD supplies real video, not existing accurate twelve-joint reference
coordinates. Those visible landmarks will be independently annotated.
AMASS projected labels describe the rendered geometry. Neither source
establishes forces, 3D clinical accuracy, or hidden-joint accuracy.

A simple-context run can test the teaching pipeline, but cannot support
a JEPA-specific claim. A JEPA run must actually load the configured
encoder and compare its features with simpler context representations.

**Continue when:** assets are available and you can prepare useful RGB
lessons and independent references. Missing appearance or human labels
is a real dependency, not a reason to substitute model predictions.

Next: [01 · Prepare source data](01_prepare_source_data.ipynb).